**1. Importação das bibliotecas**

Nesta etapa, importamos as bibliotecas necessárias para preparar os dados, criar os modelos de Machine Learning e avaliar suas previsões. Serão utilizados o `DecisionTreeRegressor` e o `RandomForestRegressor`, além das métricas RMSE e MAPE para medir os erros dos modelos.

In [1]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

**2. Carregamento e preparação dos dados**

Nesta etapa, carregamos o dataset da MecâniQA, convertemos a coluna `Data` para o formato de data e organizamos os registros em ordem cronológica. Essa organização é necessária porque estamos trabalhando com uma série temporal e precisamos garantir que os dados permaneçam na sequência correta antes do treinamento dos modelos.


In [2]:
df = pd.read_csv("/content/mecaniqa_dataset.csv")

df["Data"] = pd.to_datetime(df["Data"])

df = df.sort_values("Data").reset_index(drop=True)

print("Período dos dados:")
print(df["Data"].min(), "até", df["Data"].max())

print("\nDimensões do dataset:")
print(df.shape)

print("\nColunas:")
print(df.columns.tolist())

Período dos dados:
2024-01-01 00:00:00 até 2025-12-31 00:00:00

Dimensões do dataset:
(731, 3)

Colunas:
['Data', 'Trocas_Oleo', 'Manutencao_Motor']


**3. Criação das features temporais**

Nesta etapa, criamos atributos a partir do histórico de `Trocas_Oleo` para que os modelos possam utilizar informações anteriores na previsão.

Serão utilizadas a troca de óleo do dia anterior (`lag_1`), a troca de óleo de 7 dias atrás (`lag_7`) e a média móvel dos 7 dias anteriores (`mm_7`).

As variáveis são construídas com `shift(1)`, garantindo que a previsão de um determinado dia utilize apenas informações que já estavam disponíveis antes desse dia, evitando *data leakage*.


In [3]:
df["lag_1"] = df["Trocas_Oleo"].shift(1)

df["lag_7"] = df["Trocas_Oleo"].shift(7)

df["mm_7"] = df["Trocas_Oleo"].shift(1).rolling(7).mean()

print(df[["Data", "Trocas_Oleo", "lag_1", "lag_7", "mm_7"]].head(12))

         Data  Trocas_Oleo  lag_1  lag_7       mm_7
0  2024-01-01         11.0    NaN    NaN        NaN
1  2024-01-02          9.0   11.0    NaN        NaN
2  2024-01-03         12.0    9.0    NaN        NaN
3  2024-01-04         15.0   12.0    NaN        NaN
4  2024-01-05         25.0   15.0    NaN        NaN
5  2024-01-06         25.0   25.0    NaN        NaN
6  2024-01-07         31.0   25.0    NaN        NaN
7  2024-01-08         13.0   31.0   11.0  18.285714
8  2024-01-09         10.0   13.0    9.0  18.571429
9  2024-01-10         13.0   10.0   12.0  18.714286
10 2024-01-11         10.0   13.0   15.0  18.857143
11 2024-01-12         25.0   10.0   25.0  18.142857


**4. Separação temporal entre treino e teste**

Nesta etapa, separamos os dados respeitando a ordem cronológica. Os registros mais antigos serão utilizados para treinamento dos modelos, enquanto os registros mais recentes serão reservados para teste.

Não utilizaremos embaralhamento (*shuffle*), pois isso poderia permitir que informações do futuro influenciassem o treinamento. Dessa forma, simulamos uma situação mais próxima da aplicação real, na qual o modelo aprende com o passado e faz previsões para períodos posteriores.


In [4]:
# Removemos as linhas que não possuem todas as features necessárias
df_ml = df.dropna(subset=["Trocas_Oleo", "lag_1", "lag_7", "mm_7"]).copy()

# Define o ponto de corte: 80% para treino e 20% para teste
ponto_corte = int(len(df_ml) * 0.8)

df_treino = df_ml.iloc[:ponto_corte].copy()
df_teste = df_ml.iloc[ponto_corte:].copy()

# Define as variáveis de entrada e o alvo
features = ["lag_1", "lag_7", "mm_7"]
alvo = "Trocas_Oleo"

X_treino = df_treino[features]
y_treino = df_treino[alvo]

X_teste = df_teste[features]
y_teste = df_teste[alvo]

print("Período de treinamento:")
print(df_treino["Data"].min(), "até", df_treino["Data"].max())

print("\nPeríodo de teste:")
print(df_teste["Data"].min(), "até", df_teste["Data"].max())

print("\nQuantidade de registros:")
print("Treino:", len(df_treino))
print("Teste:", len(df_teste))

Período de treinamento:
2024-01-08 00:00:00 até 2025-08-10 00:00:00

Período de teste:
2025-08-11 00:00:00 até 2025-12-31 00:00:00

Quantidade de registros:
Treino: 568
Teste: 143


**5. Treinamento dos modelos iniciais**

Nesta etapa, treinamos os dois modelos de Machine Learning escolhidos pela equipe: `DecisionTreeRegressor` e `RandomForestRegressor`.

Neste primeiro momento, serão utilizados os hiperparâmetros padrão dos modelos. O objetivo é estabelecer uma referência inicial de desempenho antes da realização da otimização de hiperparâmetros nas próximas etapas do OAT 3.

O treinamento será realizado exclusivamente com os dados de `X_treino` e `y_treino`.


In [5]:
# Criação dos modelos com os hiperparâmetros padrão
modelo_arvore = DecisionTreeRegressor()
modelo_floresta = RandomForestRegressor()

# Treinamento utilizando somente os dados de treino
modelo_arvore.fit(X_treino, y_treino)
modelo_floresta.fit(X_treino, y_treino)

print("Modelos treinados com sucesso.")

Modelos treinados com sucesso.


**6. Geração das previsões**

Nesta etapa, utilizamos o método `predict()` para gerar as previsões dos dois modelos sobre os dados de teste.

Os modelos não tiveram acesso aos valores reais de `y_teste` durante o treinamento. Dessa forma, podemos comparar posteriormente as previsões com os valores observados e calcular as métricas de erro.


In [6]:
# Geração das previsões para o período de teste
previsoes_arvore = modelo_arvore.predict(X_teste)
previsoes_floresta = modelo_floresta.predict(X_teste)

print("Primeiras 10 previsões da Árvore de Decisão:")
print(previsoes_arvore[:10])

print("\nPrimeiras 10 previsões da Floresta Aleatória:")
print(previsoes_floresta[:10])

Primeiras 10 previsões da Árvore de Decisão:
[23. 27. 38. 20. 29. 30. 31. 22. 20. 25.]

Primeiras 10 previsões da Floresta Aleatória:
[22.15 23.13 29.21 20.12 32.   33.99 33.6  18.46 20.66 22.04]


**7. Avaliação dos modelos iniciais**

Nesta etapa, avaliamos as previsões dos modelos utilizando as métricas RMSE e MAPE.

O RMSE mede o tamanho dos erros de previsão, dando maior peso aos erros maiores. Já o MAPE representa o erro médio em termos percentuais.

Quanto menores forem os valores dessas métricas, melhor será o desempenho do modelo.


In [7]:
# Cálculo do RMSE
rmse_arvore = mean_squared_error(y_teste, previsoes_arvore) ** 0.5
rmse_floresta = mean_squared_error(y_teste, previsoes_floresta) ** 0.5

# Cálculo do MAPE
mape_arvore = mean_absolute_percentage_error(y_teste, previsoes_arvore) * 100
mape_floresta = mean_absolute_percentage_error(y_teste, previsoes_floresta) * 100

print("Árvore de Decisão:")
print(f"RMSE: {rmse_arvore:.2f}")
print(f"MAPE: {mape_arvore:.2f}%")

print("\nFloresta Aleatória:")
print(f"RMSE: {rmse_floresta:.2f}")
print(f"MAPE: {mape_floresta:.2f}%")

Árvore de Decisão:
RMSE: 4.87
MAPE: 16.64%

Floresta Aleatória:
RMSE: 4.61
MAPE: 14.74%


**8. Comparação dos modelos com os baselines**

Nesta etapa, organizamos os resultados dos modelos iniciais de Machine Learning junto aos baselines desenvolvidos nas etapas anteriores.

A comparação utiliza RMSE e MAPE, métricas definidas como critério de aceite deste encontro. Valores menores representam menor erro de previsão.


In [8]:
# Resultados dos modelos iniciais
resultados = pd.DataFrame({
    "Modelo": [
        "Naive",
        "Média Móvel (7 dias)",
        "DecisionTreeRegressor",
        "RandomForestRegressor"
    ],
    "RMSE": [
        8.92,
        8.09,
        rmse_arvore,
        rmse_floresta
    ],
    "MAPE (%)": [
        32.64,
        37.22,
        mape_arvore,
        mape_floresta
    ]
})

print(resultados.to_string(index=False))

               Modelo     RMSE  MAPE (%)
                Naive 8.920000 32.640000
 Média Móvel (7 dias) 8.090000 37.220000
DecisionTreeRegressor 4.868193 16.643089
RandomForestRegressor 4.607770 14.738961


**9. Conclusão do encontro — 30/09/2026**

Os modelos `DecisionTreeRegressor` e `RandomForestRegressor` foram treinados utilizando os hiperparâmetros padrão e respeitando a ordem cronológica dos dados.

Os dois modelos apresentaram desempenho superior aos baselines do OAT 2 nas métricas RMSE e MAPE. A `RandomForestRegressor` apresentou o melhor resultado inicial, com RMSE de 4,61 e MAPE de 14,74%.

Dessa forma, os modelos iniciais de Machine Learning demonstraram capacidade de melhorar os resultados obtidos pelos baselines anteriores, atendendo ao critério de aceite deste encontro.
